# 📘 Notebook 05: NumPy: Numerical Computing

### Course: *From Python to Deep Learning & Fine-Tuning*
**Instructor:** Ioannis Tsioulis

*Module B: Scientific Python · Notebook 1 of 3 in this module · (5 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Explain why NumPy arrays are essential for numerical and ML work
- Create, index, and slice **n-dimensional arrays** (`ndarray`)
- Use **vectorisation** to replace slow loops with fast array operations
- Apply **broadcasting** to operate on arrays of different shapes
- Perform the basic **linear-algebra** operations that underpin neural networks

> **Prerequisites:** Module A.
>
> 🔭 **Why this is a cornerstone:** every machine-learning and deep-learning library, including scikit-learn, PyTorch, and TensorFlow, represents data as arrays of numbers. A *tensor*, the central object of deep learning, is essentially a NumPy array that can also live on a GPU. Understanding NumPy means understanding the shape of all the data that flows through a model.

> ℹ️ **Setup note (browser kernel).** This course runs in a browser-based Python kernel. If an `import numpy as np` cell reports that the package is missing, run the following once, then re-run the import:

```python
import piplite
await piplite.install('numpy')
```

In [ ]:
import numpy as np
print("NumPy version:", np.__version__)

## 1. Why not just use lists?

### The problem with Python lists for numbers
Python lists are flexible but slow for numerical work, and they do not support mathematical operations element-wise. Multiplying every element of a list by 2 requires a loop. NumPy fixes both problems: its arrays are stored compactly and operate on entire blocks of numbers at once, in highly optimised C code underneath.

### The `ndarray`
NumPy's core object is the **n-dimensional array**, or `ndarray`. A 1-D array is like a vector, a 2-D array like a matrix, and higher dimensions are common in deep learning (for example, a batch of colour images is 4-D).

In [ ]:
# Create arrays from Python lists
a = np.array([1, 2, 3, 4])
b = np.array([[1, 2, 3],
              [4, 5, 6]])   # a 2-D array (matrix)

print("1-D array:", a)
print("2-D array:\n", b)
print("Shape of b:", b.shape)   # (rows, columns) = (2, 3)
print("Number of dimensions:", b.ndim)
print("Total number of elements:", b.size)
print("Data type:", a.dtype)

> 🧠 **`.shape` is the attribute you will check most often.** Almost every bug in numerical code comes from a shape mismatch. Make checking shapes a reflex.

### Convenient array constructors
NumPy can build common arrays for you, without writing out every element.

In [ ]:
print(np.zeros((2, 3)))      # a 2x3 array of zeros
print(np.ones(4))            # four ones
print(np.full((2, 2), 7))    # a 2x2 array filled with 7
print(np.eye(3))             # a 3x3 identity matrix
print(np.arange(0, 10, 2))   # like range: 0,2,4,6,8
print(np.linspace(0, 1, 5))  # 5 evenly spaced values from 0 to 1

### Reshaping arrays
The same data can be viewed in a different shape with `.reshape()`, as long as the total count matches.

In [ ]:
data = np.arange(12)          # 0..11, twelve values
print("Flat:", data)

grid = data.reshape(3, 4)     # rearrange into 3 rows of 4
print("Reshaped to 3x4:\n", grid)
print("New shape:", grid.shape)

## 2. Vectorisation: the big idea

### Intuition first
**Vectorisation** means applying an operation to a whole array at once, instead of looping element by element. You write what you mean mathematically (*'multiply the whole vector by 2'*) and NumPy executes it efficiently.

Compare the two styles:

In [ ]:
numbers = np.array([1, 2, 3, 4, 5])

# Vectorised: operate on the entire array, no loop
print(numbers * 2)        # [ 2  4  6  8 10]
print(numbers + 10)       # [11 12 13 14 15]
print(numbers ** 2)       # [ 1  4  9 16 25]
print(np.sqrt(numbers))   # square root of every element

# Element-wise operations between two arrays of the same shape
x = np.array([1, 2, 3])
y = np.array([10, 20, 30])
print(x + y)              # [11 22 33]
print(x * y)              # [10 40 90]

### Built-in summaries
NumPy arrays know how to summarise themselves with fast built-in methods.

In [ ]:
data = np.array([4, 8, 15, 16, 23, 42])
print("Sum:", data.sum())
print("Mean:", data.mean())
print("Standard deviation:", data.std().round(2))
print("Max:", data.max(), "at position", data.argmax())
print("Min:", data.min())

### Seeing the speed advantage
Vectorised code is not just shorter; it is far faster. Run this comparison.

In [ ]:
import time

size = 1_000_000
py_list = list(range(size))
np_array = np.arange(size)

# Pure Python loop
start = time.time()
result = [v * 2 for v in py_list]
print(f"Python loop: {time.time() - start:.4f} s")

# Vectorised NumPy
start = time.time()
result = np_array * 2
print(f"NumPy vectorised: {time.time() - start:.4f} s")

The NumPy version is typically tens to hundreds of times faster. At the scale of real datasets, this is the difference between seconds and hours.

## 3. Indexing and slicing

Indexing works like lists, but extends naturally to multiple dimensions. For a 2-D array we write `array[row, column]`.

In [ ]:
matrix = np.array([[10, 20, 30],
                   [40, 50, 60],
                   [70, 80, 90]])

print("Element at row 0, col 2:", matrix[0, 2])   # 30
print("Entire first row:", matrix[0])             # [10 20 30]
print("Entire second column:", matrix[:, 1])      # [20 50 80]
print("Top-left 2x2 block:\n", matrix[:2, :2])

### Boolean (mask) indexing
A powerful idea: select elements using a condition. The condition produces an array of `True`/`False`, which then picks out the matching elements. This replaces filtering loops entirely.

In [ ]:
data = np.array([3, 8, 1, 9, 4, 7])

mask = data > 5
print("The mask:", mask)        # [False  True False  True False  True]
print("Values > 5:", data[mask])  # [8 9 7]
print("Same, in one line:", data[data > 5])

# You can also assign through a mask: set every value below 5 to 0
data[data < 5] = 0
print("After zeroing small values:", data)

## 4. Broadcasting

### Intuition first
**Broadcasting** is NumPy's rule for combining arrays of *different but compatible* shapes. The smaller array is 'stretched' across the larger one without actually copying data. The simplest case you have already used: adding a single number to an array adds it to every element.

### A practical example
Suppose each row is a data point and we want to subtract the column means (a standard preprocessing step called *centering*):

In [ ]:
data = np.array([[10, 20, 30],
                 [40, 50, 60]])

col_means = data.mean(axis=0)   # mean of each column -> shape (3,)
print("Column means:", col_means)

centered = data - col_means     # broadcasting subtracts means from every row
print("Centered data:\n", centered)

> ⚠️ **Common pitfalls**
>
> - `axis=0` collapses **down the rows** (per-column result); `axis=1` collapses **across the columns** (per-row result). Beginners mix these up constantly; say it out loud each time until it sticks.
> - Broadcasting only works when shapes are compatible. Incompatible shapes raise a `ValueError`; read the shapes in the message carefully.

## 5. Linear algebra essentials

### Why this section matters most
A neural network layer computes, at its heart, a **matrix multiplication** followed by adding a bias and applying a non-linear function. The expression $\mathbf{y} = W\mathbf{x} + \mathbf{b}$ *is* a deep-learning layer. Everything below is the vocabulary of that formula.

### The dot product and matrix multiplication
The **dot product** of two vectors multiplies them element-wise and sums the result, a single number measuring how aligned they are. **Matrix multiplication** generalises this. In NumPy the `@` operator performs it.

In [ ]:
# Dot product of two vectors
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
print("Dot product:", u @ v)   # 1*4 + 2*5 + 3*6 = 32

# Matrix multiplication
W = np.array([[1, 0],
              [0, 1],
              [1, 1]])   # shape (3, 2)
x = np.array([2, 3])     # shape (2,)

print("W @ x =", W @ x)  # shape (3,), exactly a layer's linear step

> 🧠 **The golden rule of matrix multiplication shapes:** to multiply an `(m, n)` matrix by an `(n, k)` matrix, the **inner dimensions must match** (`n` and `n`). The result has shape `(m, k)`. When a deep-learning model throws a shape error, this rule is almost always what was violated.

In [ ]:
# A few more linear-algebra tools you will meet again
M = np.array([[2, 1],
              [1, 3]])
print("Transpose:\n", M.T)
print("Sum of all elements:", M.sum())
print("Max:", M.max(), " Min:", M.min())
print("Mean:", M.mean())

---
## ✏️ Exercises

### Exercise 1
Create a NumPy array of the numbers 1 to 10. Using **vectorisation** (no loop), produce a new array of their squares, then print the sum of those squares.

In [ ]:
import numpy as np
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
arr = np.arange(1, 11)
squares = arr ** 2
print(squares)
print('Sum of squares:', squares.sum())
```
</details>

### Exercise 2
Given `scores = np.array([45, 88, 72, 33, 95, 60, 51])`, use **boolean indexing** to print only the scores that are a passing grade (>= 50), and report how many passed.

In [ ]:
import numpy as np
scores = np.array([45, 88, 72, 33, 95, 60, 51])
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
passing = scores[scores >= 50]
print('Passing scores:', passing)
print('Number passing:', passing.size)
```
</details>

### Exercise 3
Create a 3 by 3 matrix with the values 1 to 9 (hint: `np.arange(1, 10).reshape(3, 3)`). Print its transpose, the sum of each column (`axis=0`), and the sum of each row (`axis=1`).

In [ ]:
import numpy as np
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
M = np.arange(1, 10).reshape(3, 3)
print('Matrix:\n', M)
print('Transpose:\n', M.T)
print('Column sums:', M.sum(axis=0))
print('Row sums:', M.sum(axis=1))
```
</details>

### Exercise 4 (a one-line neural layer)
Let `W = np.array([[0.2, 0.8], [0.5, 0.1], [0.9, 0.3]])` (shape 3 by 2) and `x = np.array([1.0, 2.0])` and `b = np.array([0.1, 0.2, 0.3])`. Compute `y = W @ x + b`. This is exactly the linear step of a neural-network layer.

In [ ]:
import numpy as np
W = np.array([[0.2, 0.8], [0.5, 0.1], [0.9, 0.3]])
x = np.array([1.0, 2.0])
b = np.array([0.1, 0.2, 0.3])
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
y = W @ x + b
print('Output of the layer:', y)
print('Shape:', y.shape)
```

*You just computed a forward pass through a single linear layer with three neurons. In Module D we will wrap exactly this operation inside PyTorch.*
</details>

## 🔑 Key takeaways

- NumPy's **`ndarray`** is the standard container for numerical data; always know its **`.shape`**.
- **Vectorisation** replaces explicit loops with whole-array operations, far faster and cleaner.
- **Indexing, slicing, and boolean masks** select data without loops; `.reshape()` changes the view.
- **Broadcasting** combines arrays of compatible shapes automatically; mind `axis=0` vs `axis=1`.
- Matrix multiplication (`@`) and the inner-dimension rule are the backbone of neural-network computation.

---
**Next:** *Notebook 06: Pandas*: loading, cleaning, and exploring real tabular datasets.

---
*© Ioannis Tsioulis. *From Python to Deep Learning & Fine-Tuning*. Prepared for educational use.*